# 📊 Разведочный анализ данных (EDA) — Страховка «Защищённый смартфон» (Сбер)

Данный ноутбук содержит полный исследовательский анализ обучающей выборки (`train.csv`) и тестовой выборки (`public_test.csv`) для задачи предсказания покупки страхового продукта.


In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

train = pd.read_csv('train.csv')
test = pd.read_csv('public_test.csv')

print(f"Размер train: {train.shape}")
print(f"Размер test: {test.shape}")
train.head()


Размер train: (6000, 29)
Размер test: (1000, 28)


,customer_id,age,gender,region,city_population,family_status,children,education,employment_type,monthly_income,...,online_payments_share,mobile_app_usage,smartphone_brand,smartphone_price,smartphone_age_months,marketing_contacts_last_year,previous_campaign_response,previous_insurance,insurance_claims,accepted
0,C100000,34,F,Казань,1270791,single,0,bachelor,employed,39963,...,0.568,4.6,Samsung,22794,5,4,0,1,0.0,0
1,C100001,32,F,Казань,1323163,single,0,master,self_employed,62651,...,0.895,2.1,Other,5792,48,0,0,1,1.0,0
2,C100002,60,F,Москва,12447781,married,3,bachelor,employed,122403,...,0.514,1.5,Samsung,50752,20,2,0,1,0.0,0
3,C100003,68,F,Санкт-Петербург,5639833,married,4,secondary,employed,39079,...,0.272,2.4,Other,7632,6,3,0,0,NaN,0
4,C100004,28,M,Краснодар,1099358,single,0,bachelor,employed,67001,...,0.959,4.2,Xiaomi,25628,5,8,1,0,NaN,0


## 1. Анализ целевой переменной `accepted`


In [2]:
target_counts = train['accepted'].value_counts()
target_pct = train['accepted'].value_counts(normalize=True) * 100

print("Распределение целевой переменной:")
for val in [0, 1]:
    print(f"Класс {val}: {target_counts[val]} ({target_pct[val]:.2f}%)")


Распределение целевой переменной:
Класс 0: 4029 (67.15%)
Класс 1: 1971 (32.85%)


## 2. Анализ пропущенных значений и доменная логика
- `loan_amount`: отсутствует ровно у тех клиентов, у которых `has_credit == 0` (сумма кредита = 0 руб.).
- `insurance_claims`: отсутствует ровно у тех клиентов, у которых `previous_insurance == 0` (нет страховых случаев, так как не было страховок).
- `credit_score`: отсутствует у части клиентов без кредитной истории.
- `mobile_app_usage`: отсутствует у клиентов без мобильного приложения (0 сессий/нед).


In [3]:
missing_train = train.isnull().sum()
missing_train = missing_train[missing_train > 0]
print("Пропуски в обучающей выборке:")
print(missing_train)


Пропуски в обучающей выборке:
loan_amount         2458
credit_score         483
mobile_app_usage     159
insurance_claims    3150
dtype: int64


## 3. Влияние ключевых признаков на конверсию в покупку страховки

### 3.1 Бренд смартфона и стоимость


In [4]:
brand_stats = train.groupby('smartphone_brand').agg(
    count=('accepted', 'count'),
    conversion=('accepted', 'mean'),
    avg_price=('smartphone_price', 'mean'),
    avg_income=('monthly_income', 'mean')
).sort_values(by='conversion', ascending=False)
print("Статистика по брендам смартфонов:")
print(brand_stats)


Статистика по брендам смартфонов:
                  count  conversion     avg_price     avg_income
smartphone_brand                                                
Apple              1097    0.810392  99356.776664  145279.706472
Google              138    0.521739  61830.065217  109261.659420
Samsung            1367    0.455011  53648.678127   98769.243599
OnePlus             136    0.352941  39782.625000   92732.235294
Huawei              658    0.145897  25726.828267   72418.779635
Xiaomi             1931    0.119109  21910.351113   66076.981875
Other               673    0.020802  10551.508172   56248.268945


### 3.2 Региональное распределение и образование


In [5]:
print("Конверсия по регионам:")
print(train.groupby('region')['accepted'].agg(['count', 'mean']).sort_values(by='mean', ascending=False))

print("\nКонверсия по уровню образования:")
print(train.groupby('education')['accepted'].agg(['count', 'mean']).sort_values(by='mean', ascending=False))


Конверсия по регионам:
                 count      mean
region                          
Москва            1146  0.577661
Санкт-Петербург    709  0.461213
Казань             407  0.272727
Екатеринбург       424  0.250000
Новосибирск        427  0.245902
Самара             457  0.242888
Уфа                371  0.237197
Ростов-на-Дону     390  0.235897
Омск               438  0.232877
Краснодар          366  0.226776
Нижний Новгород    453  0.225166
Челябинск          412  0.199029

Конверсия по уровню образования:
           count      mean
education                 
phd           81  0.580247
master      1115  0.467265
bachelor    2903  0.364795
secondary   1901  0.180957


## 4. Корреляционный анализ числовых признаков с целевой переменной


In [6]:
num_cols = train.select_dtypes(include=[np.number]).columns.drop('accepted')
corrs = train[num_cols].apply(lambda c: c.corr(train['accepted'])).sort_values(ascending=False)

print("Корреляция признаков с целевой переменной 'accepted':")
print(corrs)


Корреляция признаков с целевой переменной 'accepted':
smartphone_price                     0.708694
monthly_income                       0.621465
number_of_card_transactions_month    0.595503
credit_score                         0.514668
previous_insurance                   0.415547
average_monthly_balance              0.385650
number_of_bank_products              0.354218
online_payments_share                0.343283
previous_campaign_response           0.342909
marketing_contacts_last_year         0.316218
mobile_app_usage                     0.306616
city_population                      0.291776
loan_amount                          0.283538
owns_car                             0.275260
owns_house                           0.191754
has_credit                           0.134544
insurance_claims                     0.062359
years_with_bank                      0.043511
children                            -0.045659
age                                 -0.138835
smartphone_age_months     

## 5. Выводы EDA и гипотезы для Feature Engineering
1. **Стоимость смартфона (`smartphone_price`)** — самый сильный предиктор (корреляция 0.71). Клиенты с дорогими телефонами (Apple, Samsung флагманы) значительно чаще покупают страховку экрана.
2. **Доход и баланс (`monthly_income`, `average_monthly_balance`)** — высокая платежеспособность коррелирует с защитой активов.
3. **Отношение стоимости смартфона к доходу (`price_to_income`)** — важный финансовый показатель ценности гаджета для пользователя.
4. **Предыдущий опыт страхования и маркетинговый отклик (`previous_insurance`, `previous_campaign_response`)** — клиенты с опытом страхования охотнее покупают защиту.
5. **Цифровое поведение (`online_payments_share`, `mobile_app_usage`)** — цифровые активные пользователи банка чаще реагируют на диджитал-продукты.
